In [4]:
import requests
import pandas as pd
from tqdm import tqdm
from datetime import datetime, UTC
import time
import random

CITY_IDS = {
    "new_delhi":  49,
    "mumbai":     201,
    "bangalore":  105,
    "hyderabad":  8,
    "chennai":    280,
    "pune":       205,
    "ahmedabad":  51,
    "jaipur":     269,
    "lucknow":    320,
    "chandigarh": 42,
    "nagpur":     202,
    "indore":     164,
    "coimbatore": 281,
    "surat":      66,
    "kolkata":    338,
}

# 30 models across all segments — budget, mid, premium
MODELS = [
    # Maruti — highest volume on CarDekho
    "maruti-swift", "maruti-baleno", "maruti-dzire",
    "maruti-wagon-r", "maruti-alto", "maruti-vitara-brezza",
    "maruti-ertiga", "maruti-ciaz", "maruti-s-cross",

    # Hyundai
    "hyundai-i20", "hyundai-grand-i10", "hyundai-creta",
    "hyundai-verna", "hyundai-venue",

    # Tata
    "tata-nexon", "tata-tiago", "tata-harrier",
    "tata-punch", "tata-safari",

    # Honda
    "honda-city", "honda-amaze", "honda-jazz",

    # Others — high volume
    "kia-seltos", "kia-sonet",
    "mahindra-scorpio", "mahindra-xuv500", "mahindra-thar",
    "toyota-innova", "renault-kwid",
]
# 30 models, all high-inventory → much better hit rate

TARGET    = 10000
PAGES_PER_COMBO = 5   # ~20 listings × 5 pages = up to 100 per model+city combo
                      # 30 models × 15 cities × ~20 avg = ~9,000 realistically

In [5]:
def fetch_page(city_name, city_id, page, model=None):
    url = "https://listing.cardekho.com/api/v1/srp-listings"

    # searchstring changes based on whether model is specified
    if model:
        searchstring = f"used-{model}-cars-in-{city_name.replace('_', '-')}"
    else:
        searchstring = f"used-cars-in-{city_name.replace('_', '-')}"

    params = {
        "searchstring": searchstring,
        "cityId":       city_id,
        "page":         page,
        "count":        20,
        "platform":     "web",
        "lang_code":    "en",
    }

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept":   "application/json",
        "Referer":  f"https://www.cardekho.com/{searchstring}",
    }

    try:
        res = requests.get(url, headers=headers, params=params, timeout=10)
        res.raise_for_status()
        data = res.json()
    except Exception as e:
        print(f"  [ERROR] {city_name}/{model} p{page}: {e}")
        return []

    listings = data.get("data", {}).get("cars", [])

    parsed = []
    for car in listings:
        if not car.get("price") or not car.get("myear"):
            continue
        parsed.append({
            "brand":        car.get("oem"),
            "model":        car.get("model"),
            "variant":      car.get("variantName"),
            "year":         car.get("myear"),
            "fuel":         car.get("ft"),
            "transmission": car.get("tt"),
            "km":           car.get("km"),
            "body_type":    car.get("bt"),          
            "owner":        car.get("owner"),
            "owner_slug":   car.get("ownerSlug"),
            "seller_type":  car.get("utype"), 
            "locality":     car.get("loc"),
            "price":        car.get("price"),
            "price_text":   car.get("formattedPrice"),
            "search_city":  city_name,
            "actual_city":  car.get("city"),
            "listing_url":  "https://www.cardekho.com" + (car.get("vlink") or ""),
            "scraped_at":   datetime.now(UTC).isoformat(),
})

    return parsed

In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

all_data   = []
seen_urls  = set()
lock       = threading.Lock()  # thread-safe writes to shared data

combos = [
    (city_name, city_id, model)
    for model in MODELS
    for city_name, city_id in CITY_IDS.items()
]

print(f"Total combos: {len(combos)} | Target: {TARGET}\n")

def scrape_combo(city_name, city_id, model):
    """Scrape all pages for one model+city combo. Returns list of new unique cars."""
    local_results  = []
    consecutive_dups = 0

    for page in range(1, PAGES_PER_COMBO + 1):
        cars = fetch_page(city_name, city_id, page, model)

        if not cars:
            break

        new_this_page = 0
        for car in cars:
            url = car["listing_url"]
            with lock:
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    local_results.append(car)
                    new_this_page += 1

        if new_this_page == 0:
            consecutive_dups += 1
            if consecutive_dups >= 2:
                break
        else:
            consecutive_dups = 0

        time.sleep(random.uniform(0.1, 0.2))  # much shorter delay

    return local_results


# Run 10 combos in parallel
with ThreadPoolExecutor(max_workers=7) as executor:
    futures = {
        executor.submit(scrape_combo, city_name, city_id, model): (model, city_name)
        for city_name, city_id, model in combos
    }

    with tqdm(total=len(combos), desc="Scraping") as pbar:
        for future in as_completed(futures):
            model, city_name = futures[future]
            try:
                results = future.result()
                all_data.extend(results)

                if len(all_data) >= TARGET:
                    executor.shutdown(wait=False, cancel_futures=True)

                if len(all_data) % 500 < 20 and len(all_data) > 0:
                    print(f"  → {len(all_data)} unique | last: {model} in {city_name}")

            except Exception as e:
                print(f"  [ERROR] {model} in {city_name}: {e}")

            pbar.update(1)

# Trim to target
all_data = all_data[:TARGET]
print(f"\nDone. Total unique listings: {len(all_data)}")

Total combos: 435 | Target: 10000



Scraping: 100%|██████████████████████████████████████████████████████████████████████| 435/435 [03:31<00:00,  2.06it/s]


Done. Total unique listings: 305


In [7]:
df = pd.DataFrame(all_data)
df

,brand,model,variant,year,fuel,transmission,km,body_type,owner,owner_slug,seller_type,locality,price,price_text,search_city,actual_city,listing_url,scraped_at
0,Ford,Ford Ecosport,1.5 TDCi Titanium BSIV,2016,Diesel,Manual,"1,17,300",SUV,1,first-owner,Individual,Pallikaranai,535000,₹5.35 Lakh,chennai,Chennai,https://www.cardekho.com/used-car-details/used...,2026-05-01T15:49:51.147646+00:00
1,MG,MG Astor,Sharp CVT BSVI,2022,Petrol,Automatic,"25,000",SUV,1,first-owner,Dealer,nungambakkam,1365000,₹13.65 Lakh,chennai,Chennai,https://www.cardekho.com/used-car-details/used...,2026-05-01T15:49:51.147670+00:00
2,Volkswagen,Volkswagen Tiguan,2.0 TSI Elegance,2023,Petrol,Automatic,"34,965",SUV,1,first-owner,Dealer,medavakkam,2490000,₹24.90 Lakh,chennai,Chennai,https://www.cardekho.com/used-car-details/used...,2026-05-01T15:49:51.147679+00:00
3,Hyundai,Hyundai i20,1.2 Asta Option,2017,Petrol,Manual,"66,272",Hatchback,1,first-owner,Dealer,Sholinganallur,547000,₹5.47 Lakh,chennai,Chennai,https://www.cardekho.com/buy-used-car-details/...,2026-05-01T15:49:51.147686+00:00
4,Honda,Honda Jazz,1.2 VX i VTEC,2018,Petrol,Manual,"30,000",Hatchback,1,first-owner,Individual,Kotturpuram,570000,₹5.70 Lakh,chennai,Chennai,https://www.cardekho.com/used-car-details/used...,2026-05-01T15:49:51.147693+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300,Renault,Renault KWID,1.0 RXT Opt,2019,Petrol,Manual,"70,807",Hatchback,3,third-owner,Dealer,peelamedu,310000,₹3.10 Lakh,coimbatore,Coimbatore,https://www.cardekho.com/buy-used-car-details/...,2026-05-01T15:50:01.384465+00:00
301,Renault,Renault KWID,1.0 RXT,2023,Petrol,Manual,"30,848",Hatchback,1,first-owner,Dealer,peelamedu,423000,₹4.23 Lakh,coimbatore,Coimbatore,https://www.cardekho.com/buy-used-car-details/...,2026-05-01T15:50:01.384497+00:00
302,Renault,Renault KWID,1.0 RXT Opt,2019,Petrol,Manual,"48,350",Hatchback,1,first-owner,Dealer,peelamedu,364000,₹3.64 Lakh,coimbatore,Coimbatore,https://www.cardekho.com/buy-used-car-details/...,2026-05-01T15:50:01.384511+00:00
303,Renault,Renault KWID,Climber 1.0 MT,2018,Petrol,Manual,"33,107",Hatchback,1,first-owner,Dealer,peelamedu,376000,₹3.76 Lakh,coimbatore,Coimbatore,https://www.cardekho.com/buy-used-car-details/...,2026-05-01T15:50:01.384523+00:00


In [9]:
df.to_csv("../data/raw/cars_scraped.csv", index=False)

In [10]:
df.head()

,brand,model,variant,year,fuel,transmission,km,body_type,owner,owner_slug,seller_type,locality,price,price_text,search_city,actual_city,listing_url,scraped_at
0,Ford,Ford Ecosport,1.5 TDCi Titanium BSIV,2016,Diesel,Manual,"1,17,300",SUV,1,first-owner,Individual,Pallikaranai,535000,₹5.35 Lakh,chennai,Chennai,https://www.cardekho.com/used-car-details/used...,2026-05-01T15:49:51.147646+00:00
1,MG,MG Astor,Sharp CVT BSVI,2022,Petrol,Automatic,"25,000",SUV,1,first-owner,Dealer,nungambakkam,1365000,₹13.65 Lakh,chennai,Chennai,https://www.cardekho.com/used-car-details/used...,2026-05-01T15:49:51.147670+00:00
2,Volkswagen,Volkswagen Tiguan,2.0 TSI Elegance,2023,Petrol,Automatic,"34,965",SUV,1,first-owner,Dealer,medavakkam,2490000,₹24.90 Lakh,chennai,Chennai,https://www.cardekho.com/used-car-details/used...,2026-05-01T15:49:51.147679+00:00
3,Hyundai,Hyundai i20,1.2 Asta Option,2017,Petrol,Manual,"66,272",Hatchback,1,first-owner,Dealer,Sholinganallur,547000,₹5.47 Lakh,chennai,Chennai,https://www.cardekho.com/buy-used-car-details/...,2026-05-01T15:49:51.147686+00:00
4,Honda,Honda Jazz,1.2 VX i VTEC,2018,Petrol,Manual,"30,000",Hatchback,1,first-owner,Individual,Kotturpuram,570000,₹5.70 Lakh,chennai,Chennai,https://www.cardekho.com/used-car-details/used...,2026-05-01T15:49:51.147693+00:00
